In [ ]:
%cd /content
!test -d in-tfm || git clone -q https://github.com/narang99/in-tfm.git
%cd /content/in-tfm
!git fetch -q && git checkout -q feat/bucket-report && git log --oneline -1
!python -c "import torch, numpy, sklearn; print('before', torch.__version__, numpy.__version__, sklearn.__version__)"
!pip install -q uv
!uv pip install --system -q --no-deps -e .
!uv pip install --system -q captum jaxtyping lxt nnsight pydantic-settings rad-dino hdbscan pydicom kornia
!uv pip install --system -q --no-deps "torch-lucent @ git+https://github.com/greentfrapp/lucent@dev"
!python -c "import torch, numpy, sklearn; print('after', torch.__version__, numpy.__version__, sklearn.__version__)"

/content/in-tfm
2e35a65 (HEAD -> feat/bucket-report, origin/feat/bucket-report) feat: each map is also drawn at four times the size and linked, to zoom into a crowded corner
before 2.11.0+cu130 2.1.3 1.6.1
after 2.11.0+cu130 2.1.3 1.6.1
[first run lacked pydicom and kornia; stage 1 failed with ModuleNotFoundError: No module named 'pydicom'. Installed them, now in the install line above]
2.11.0+cu130 True


In [ ]:
!python -c "import torch; assert torch.cuda.is_available() and '+cu' in torch.__version__, torch.__version__; print(torch.cuda.get_device_name())"
!python -c "import in_tfm, lucent, captum, nnsight, hdbscan, networkx; print('imports ok', in_tfm.__file__)"

Tesla T4
imports ok /content/in-tfm/src/in_tfm/__init__.py


In [ ]:
%cd /content/in-tfm
!mkdir -p data
!test -f /content/inception_pool.tar || curl -sSf -H "Authorization: Bearer $GTOKEN" -o /content/inception_pool.tar 'https://storage.googleapis.com/storage/v1/b/pj-okc/o/inception_pool.tar?alt=media'
!tar --warning=no-unknown-keyword -tf /content/inception_pool.tar | grep -E '^/|(^|/)\.\.(/|$)' | head; echo "unsafe paths above (should be empty)"
!tar --warning=no-unknown-keyword -tf /content/inception_pool.tar | head -3
!tar --warning=no-unknown-keyword -xf /content/inception_pool.tar -C data && find data -name '._*' -delete
!ls data/inception_pool | wc -l; find data/inception_pool -name '*.jpg' | wc -l



/content/in-tfm
unsafe paths above (should be empty)
._inception_pool
inception_pool/
tar: inception_pool/._n02795169
123
12300


In [ ]:
%cd /content/in-tfm
!time python -u scripts/collect_hit_space.py --config configs/inception_v1_mixed4e_peaks.yaml --inception.image-dir data/inception_pool --out-dir runs/mixed4e_55; echo "exit $?"
!ls -la runs/mixed4e_55/space



/content/in-tfm
Downloading: "https://github.com/ProGamerGov/pytorch-old-tensorflow-models/raw/master/inception5h.pth" to /root/.cache/torch/hub/checkpoints/inception5h.pth

  0% 0.00/27.0M [00:00<?, ?B/s]
100% 27.0M/27.0M [00:00<00:00, 429MB/s]

scan:   0% 0/125 [00:00<?, ?it/s]
scan:   1% 1/125 [00:00<01:12,  1.72it/s]
scan:   2% 2/125 [00:00<00:59,  2.08it/s]
scan:   2% 3/125 [00:01<00:55,  2.20it/s]
scan:   3% 4/125 [00:01<00:58,  2.07it/s]
scan:   4% 5/125 [00:02<00:54,  2.20it/s]
scan:   5% 6/125 [00:02<00:52,  2.27it/s]
scan:   6% 7/125 [00:03<00:50,  2.33it/s]
scan:   6% 8/125 [00:03<00:49,  2.38it/s]
scan:   7% 9/125 [00:04<00:54,  2.13it/s]
scan:   8% 10/125 [00:04<00:51,  2.23it/s]
scan:   9% 11/125 [00:04<00:50,  2.27it/s]
scan:  10% 12/125 [00:05<00:48,  2.35it/s]
scan:  10% 13/125 [00:05<00:46,  2.42it/s]
scan:  11% 14/125 [00:06<00:44,  2.47it/s]
scan:  12% 15/125 [00:06<00:43,  2.50it/s]
scan:  13% 16/125 [00:06<00:43,  2.48it/s]
scan:  14% 17/125 [00:07<00:42,  2.52i

In [ ]:
%cd /content/in-tfm
!python -u scripts/hit_buckets.py runs/mixed4e_55/space; echo "exit $?"



14240 hits, 41 clusters, 2606 non-empty buckets, 225 with >= 10 hits
    2603  cluster 0 >= 0.7
    2162  cluster 0 >= 0.8
    1535  cluster 0 >= 0.9
     844  cluster 3 >= 0.7
     744  cluster 3 >= 0.8
     647  cluster 1 >= 0.7
     579  cluster 3 >= 0.9
     553  clusters 0 + 4
     535  clusters 0 + 15
     496  cluster 1 >= 0.8
     381  cluster 5 >= 0.7
     344  cluster 5 >= 0.8
     305  clusters 1 + 2
     283  cluster 5 >= 0.9
     282  cluster 1 >= 0.9
     236  clusters 0 + 3 + 4
     223  cluster 14 >= 0.7
     192  clusters 0 + 18
     178  cluster 4 >= 0.7
     178  cluster 12 >= 0.7
     175  cluster 6 >= 0.7
     170  cluster 9 >= 0.7
     168  cluster 15 >= 0.7
     161  cluster 13 >= 0.7
     155  cluster 12 >= 0.8
     154  cluster 6 >= 0.8
     154  cluster 14 >= 0.8
     144  cluster 8 >= 0.7
     143  cluster 2 >= 0.7
     141  cluster 13 >= 0.8
     140  cluster 19 >= 0.7
     140  clusters 0 + 3
     137  cluster 16 >= 0.7
     136  cluster 9 >= 0.8
     134

In [ ]:
%cd /content/in-tfm
!time python -u scripts/bucket_report.py --config configs/inception_v1_mixed4e_peaks.yaml --inception.image-dir data/inception_pool --out-dir runs/mixed4e_55; echo "exit $?"



[layout] 56.95s
1655 sampled hits, 6 neighbours each
[bucket pages] 774.20s
bucket index -> runs/mixed4e_55/bucket_report/index.html

real	14m3.040s
user	14m7.787s
sys	0m30.771s
exit 0
__DONE__


In [ ]:
%cd /content/in-tfm
!time python -u scripts/run_neuron_report.py --config configs/inception_v1_mixed4e_peaks.yaml --hits.method elbow --out-dir reports/inception_peaks_elbow; echo "exit $?"



[load samples] 0.23s
[load model] 0.09s

scan: 100%|██████████| 125/125 [00:49<00:00,  2.54it/s]
[pass 1: scan] 49.17s
scanned 2000 samples, activations (2000, 196, 1), 392000/392000 positions unpadded
[neuron 55 positive] elbow threshold=203.2316, 14240 hits beyond it, 14240 kept

gather: 100%|██████████| 61/61 [00:24<00:00,  2.54it/s]
[pass 2: gather] 24.06s
gathered 14240 positions; pass 2 matches pass 1 to 0.00e+00
[neuron 55 positive: cluster] 674.03s
[neuron 55 positive] 41 clusters, 10304 of 14240 hits too mixed to show
[neuron 55 positive: write mixed hits] 10.00s
[neuron 55 positive] mixed hits -> reports/inception_peaks_elbow/neuron_55/mixed_hits.html

100%|██████████| 10/10 [00:03<00:00,  3.04it/s]

100%|██████████| 10/10 [00:03<00:00,  3.05it/s]

100%|██████████| 10/10 [00:03<00:00,  3.01it/s]

100%|██████████| 10/10 [00:03<00:00,  3.05it/s]

100%|██████████| 10/10 [00:03<00:00,  3.07it/s]

100%|██████████| 10/10 [00:03<00:00,  3.00it/s]

100%|██████████| 10/10 [00:03<00:

In [ ]:
%cd /content/in-tfm
import os, re, pathlib, html.parser
r = pathlib.Path('runs/mixed4e_55/bucket_report')
pages = list(r.rglob('*.html'))
print('html pages', len(pages))
maps = [p.name for p in r.glob('cluster*_*.png')]
print('map files', len(maps), 'huge', sum(n.endswith('_huge.png') for n in maps))
print('hit_ card dirs', len([p for p in r.rglob('hit_*') if p.is_dir()]))
class L(html.parser.HTMLParser):
    def __init__(s): super().__init__(); s.refs=[]
    def handle_starttag(s, t, a): s.refs += [v for k, v in a if k in ('src','href') and v]
total, missing = 0, []
for p in pages:
    l = L(); l.feed(p.read_text())
    for v in l.refs:
        if re.match(r'^(https?:|mailto:|#|data:)', v): continue
        total += 1
        if not (p.parent / v.split('#')[0].split('?')[0]).exists(): missing.append((str(p), v))
print('local refs', total, 'missing', len(missing), missing[:5])
print('neuron report', os.path.exists('reports/inception_peaks_elbow/neuron_55/index.html'))



/content/in-tfm
html pages 226
map files 450 huge 225
hit_ card dirs 1113
local refs 8436 missing 0 []
neuron report True


In [ ]:
%cd /content/in-tfm
!tar -cf /content/outputs.tar runs/mixed4e_55/bucket_report runs/mixed4e_55/space/buckets.json reports/inception_peaks_elbow && ls -lh /content/outputs.tar



/content/in-tfm
-rw-r--r-- 1 root root 777M Oct 24 05:12 /content/outputs.tar


In [ ]:
%cd /content/in-tfm
!time python -u scripts/run_neuron_report.py --config configs/inception_v1_mixed4e_peaks.yaml --hits.method bucketed --out-dir reports/inception_peaks_bucketed; echo "exit $?"



[load samples] 0.23s
[load model] 0.10s

scan: 100%|██████████| 125/125 [00:49<00:00,  2.55it/s]
[pass 1: scan] 49.01s
scanned 2000 samples, activations (2000, 196, 1), 392000/392000 positions unpadded
[neuron 55 positive] bucketed threshold=0.0000, 176329 hits beyond it, 20000 kept

gather: 100%|██████████| 119/119 [00:47<00:00,  2.53it/s]
[pass 2: gather] 47.10s
gathered 20000 positions; pass 2 matches pass 1 to 0.00e+00
[neuron 55 positive: cluster] 1419.50s
[neuron 55 positive] 12 clusters, 12693 of 20000 hits too mixed to show
[neuron 55 positive: write mixed hits] 9.17s
[neuron 55 positive] mixed hits -> reports/inception_peaks_bucketed/neuron_55/mixed_hits.html

100%|██████████| 10/10 [00:03<00:00,  2.94it/s]

100%|██████████| 10/10 [00:03<00:00,  3.00it/s]

100%|██████████| 10/10 [00:03<00:00,  3.00it/s]

100%|██████████| 10/10 [00:03<00:00,  2.98it/s]

100%|██████████| 10/10 [00:03<00:00,  3.04it/s]

100%|██████████| 10/10 [00:03<00:00,  3.06it/s]

100%|██████████| 10/10 [00